In [0]:
from pyspark.sql import functions as F

# 1. Load cleaned dataset from Databricks Unity Catalog Volume
DATA_PATH = "/Volumes/cs483eviction/raw/health_atlas/cook_county_evictions_cleaned_2001_2015.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(DATA_PATH)
    .withColumn("cofips", F.lpad(F.col("cofips").cast("string"), 5, "0"))
    .withColumn("geoid", F.lpad(F.col("geoid").cast("string"), 11, "0"))
    .withColumn("is_chicago_tract", F.col("is_chicago_tract").cast("boolean"))
)

# 2. Scoped key feature subsets & full indicator labels
evict_cols = ["filings", "filing_rate", "judgements", "judgement_rate"]
cha_cols = [
    "cha_INC_2020-2024",
    "cha_POV_2020-2024",
    "cha_HBU_2020-2024",
    "cha_VAC_2020-2024",
    "cha_SVI_2022",
]

FEATURE_LABELS = {
    "cha_INC_2020-2024": "Median Household Income ($, 2020-2024)",
    "cha_POV_2020-2024": "Poverty Rate (%, 2020-2024)",
    "cha_HBU_2020-2024": "Housing Cost Burden (%, 2020-2024)",
    "cha_VAC_2020-2024": "Housing Vacancy Rate (%, 2020-2024)",
    "cha_SVI_2022": "Social Vulnerability Index (0-100, 2022)",
}


def summarize_numeric(sdf, cols, label_map=None):
    """Compute descriptive statistics in Spark and map column codes to readable labels."""
    label_map = label_map or {}
    rows = []
    for c in cols:
        col_ref = f"`{c}`"
        display_name = label_map.get(c, c)
        stats = sdf.agg(
            F.lit(display_name).alias("feature"),
            F.count(F.col(c)).alias("count"),
            F.round(F.mean(F.col(c)), 2).alias("mean"),
            F.round(F.stddev(F.col(c)), 2).alias("std"),
            F.round(F.min(F.col(c)), 2).alias("min"),
            F.round(F.expr(f"percentile({col_ref}, 0.25)"), 2).alias("q1_25%"),
            F.round(F.expr(f"percentile({col_ref}, 0.50)"), 2).alias("median_50%"),
            F.round(F.expr(f"percentile({col_ref}, 0.75)"), 2).alias("q3_75%"),
            F.round(F.max(F.col(c)), 2).alias("max"),
            F.round(
                F.expr(f"percentile({col_ref}, 0.75)") - F.expr(f"percentile({col_ref}, 0.25)"),
                2,
            ).alias("IQR"),
            F.round(F.skewness(F.col(c)), 2).alias("skew"),
        )
        rows.append(stats)

    summary_df = rows[0]
    for r in rows[1:]:
        summary_df = summary_df.unionByName(r)
    return summary_df


# 3. Eviction panel summary (all Cook County tract-years, n = 19,725)
evict_summary = summarize_numeric(df, evict_cols)
print("=== Eviction Panel Summary (Cook County, 2001-2015) ===")
display(evict_summary)

# 4. Neighborhood context summary with full indicator names (unique Chicago tracts, n = 789)
chi_tracts = df.filter(F.col("is_chicago_tract")).dropDuplicates(["geoid"])
cha_summary = summarize_numeric(chi_tracts, cha_cols, label_map=FEATURE_LABELS)
print("=== Neighborhood Features Summary (Unique Chicago Tracts) ===")
display(cha_summary)

In [0]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

DATA_PATH = "/Volumes/cs483eviction/raw/health_atlas/cook_county_evictions_cleaned_2001_2015.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(DATA_PATH)
    .withColumn("cofips", F.lpad(F.col("cofips").cast("string"), 5, "0"))
    .withColumn("geoid", F.lpad(F.col("geoid").cast("string"), 11, "0"))
    .withColumn("is_chicago_tract", F.col("is_chicago_tract").cast("boolean"))
)

plt.style.use("ggplot")

# ==============================================================================
# FIGURE 1: Annual Eviction Volume & Mean Rate + Chicago vs. Suburban Box Plots
# ==============================================================================
yearly_pdf = (
    df.groupBy("year")
    .agg(
        F.sum("filings").alias("total_filings"),
        F.sum("judgements").alias("total_judgements"),
        F.mean("filing_rate").alias("mean_filing_rate"),
    )
    .orderBy("year")
    .toPandas()
)

rates_pdf = df.select("year", "is_chicago_tract", "filing_rate", "judgement_rate").toPandas()

fig, axes = plt.subplots(1, 2, figsize=(15, 5.8), gridspec_kw={"width_ratios": [1.35, 1]})
fig.suptitle("Cook County Eviction Trends (2001–2015) & Geographic Comparison", fontsize=14, fontweight="bold", y=0.98)

x = np.arange(len(yearly_pdf))
w = 0.38
ax1 = axes[0]
ax1.bar(x - w / 2, yearly_pdf["total_filings"] / 1000, width=w, color="#2b5c8f", label="Total Filings (thousands)")
ax1.bar(x + w / 2, yearly_pdf["total_judgements"] / 1000, width=w, color="#e67e22", label="Total Judgements (thousands)")
ax1.axvline(4.5, color="#c0392b", linestyle="--", linewidth=1.3, alpha=0.85)
ax1.text(4.65, 44.5, "2006 Judgement\nReporting Shift", color="#c0392b", fontsize=8.5, fontweight="bold", va="top")

ax1_r = ax1.twinx()
ax1_r.plot(x, yearly_pdf["mean_filing_rate"], color="#1e8449", marker="o", linewidth=2.2, label="Mean Filing Rate (%)")
ax1_r.set_ylabel("Mean Filing Rate (%)", color="#1e8449", fontweight="bold")
ax1_r.tick_params(axis="y", labelcolor="#1e8449")
ax1_r.set_ylim(0, 7.5)
ax1_r.grid(False)

ax1.set_xticks(x)
ax1.set_xticklabels(yearly_pdf["year"], rotation=45)
ax1.set_ylabel("Annual Count (Thousands)", fontweight="bold")
ax1.set_title("Annual Eviction Volume & Mean Filing Rate", fontsize=11.5, fontweight="bold")
ax1.set_ylim(0, 50)

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax1_r.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper right", fontsize=8.5, frameon=True)

# Box plots
ax2 = axes[1]
chi_fr = rates_pdf.loc[rates_pdf["is_chicago_tract"], "filing_rate"].values
sub_fr = rates_pdf.loc[~rates_pdf["is_chicago_tract"], "filing_rate"].values
chi_jr = rates_pdf.loc[(rates_pdf["is_chicago_tract"]) & (rates_pdf["year"] >= 2006), "judgement_rate"].values
sub_jr = rates_pdf.loc[(~rates_pdf["is_chicago_tract"]) & (rates_pdf["year"] >= 2006), "judgement_rate"].values

bp = ax2.boxplot(
    [chi_fr, sub_fr, chi_jr, sub_jr],
    positions=[1, 1.45, 2.5, 2.95],
    widths=0.38,
    patch_artist=True,
    showfliers=False,
    medianprops={"color": "#111111", "linewidth": 1.8},
)
colors = ["#2b5c8f", "#e67e22", "#2b5c8f", "#e67e22"]
for patch, c in zip(bp["boxes"], colors):
    patch.set_facecolor(c)

ax2.set_xticks([1.225, 2.725])
ax2.set_xticklabels(["Filing Rate\n(2001–2015)", "Judgement Rate\n(2006–2015)"])
ax2.set_title("Rate Distributions: Chicago vs. Suburban Cook", fontsize=11.5, fontweight="bold")
ax2.set_xlabel("Outliers hidden for readability (max filing rate = 150%)", fontsize=8.5)
ax2.set_ylabel("Rate per 100 Renter Households (%)", fontweight="bold")
ax2.set_ylim(0, 15.5)
ax2.legend([bp["boxes"][0], bp["boxes"][1]], ["City of Chicago", "Suburban Cook"], loc="upper right", frameon=True)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

# ==============================================================================
# FIGURE 2: Tract-Level Scatter Plots with Trendlines 
# ==============================================================================
chi_df = df.filter(F.col("is_chicago_tract"))

tract_filings = chi_df.groupBy("geoid").agg(
    F.mean("filings").alias("avg_filings"),
    F.mean("filing_rate").alias("avg_filing_rate"),
    F.first(F.col("`cha_INC_2020-2024`")).alias("cha_inc"),
    F.first(F.col("`cha_POV_2020-2024`")).alias("cha_pov"),
    F.first(F.col("`cha_HBU_2020-2024`")).alias("cha_hbu"),
    F.first(F.col("`cha_VAC_2020-2024`")).alias("cha_vac"),
    F.first(F.col("`cha_SVI_2022`")).alias("cha_svi"),
)

tract_judg = (
    chi_df.filter(F.col("year") >= 2006)
    .groupBy("geoid")
    .agg(
        F.mean("judgements").alias("avg_judgements_06_15"),
        F.mean("judgement_rate").alias("avg_judgement_rate_06_15"),
    )
)

chi_agg_pdf = tract_filings.join(tract_judg, on="geoid", how="inner").toPandas()

fig, axes = plt.subplots(2, 2, figsize=(13.5, 10))
fig.suptitle("Tract-Level Eviction Filing Rate vs. Neighborhood Context (Chicago, n = 789 Tracts)", fontsize=14, fontweight="bold", y=0.98)

scatter_specs = [
    ("cha_inc", "Median Household Income ($, 2020–2024)", "#2b5c8f"),
    ("cha_pov", "Poverty Rate (%, 2020–2024)", "#c0392b"),
    ("cha_hbu", "Housing Cost Burden (%, 2020–2024)", "#d35400"),
    ("cha_svi", "Social Vulnerability Index (0–100, 2022)", "#6c3483"),
]

for ax, (col, xlabel, color) in zip(axes.flat, scatter_specs):
    x_vals = chi_agg_pdf[col].values
    y_vals = chi_agg_pdf["avg_filing_rate"].values
    r_val = np.corrcoef(x_vals, y_vals)[0, 1]
    slope, intercept = np.polyfit(x_vals, y_vals, 1)
    x_line = np.linspace(x_vals.min(), x_vals.max(), 100)

    ax.scatter(x_vals, y_vals, alpha=0.45, s=22, color=color, edgecolors="none")
    ax.plot(x_line, slope * x_line + intercept, color="#111111", linewidth=2)
    ax.set_xlabel(xlabel, fontweight="bold")
    ax.set_ylabel("Avg Annual Filing Rate % (2001–2015)", fontweight="bold")
    ax.set_title(f"{xlabel.split(' (')[0]} (r = {r_val:+.2f})", fontsize=11, fontweight="bold")
    ax.set_ylim(0, 20)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

#FINDINGS:  Filling Rates went from a low 4% to a peak of 5.65% during 2008-2009 financial crisis. While Chicago tracts generate higher raw eviction counts, Suburban Cook County has higher normalized eviction intensity across both filing rates  and judgement rates. Tracts in the highest SVI quartile averaged 6.47% annual filing rates and 2.37% judgement rates which is nearly 3× the rates of the lowest quartile. Median Household Income and Social Vulnerability Index are strongly collinear, as are Poverty Rate and Income, and Filings with Judgements.
